# OralIQ — Core Concept Demo
### How AI Generates Oral Exam Questions from a Grading Rubric

This notebook shows the **one essential idea** behind OralIQ in ~20 lines of Python: give Claude a grading rubric and it autonomously generates oral exam questions aligned to each criterion.

The full app (the HTML file) wraps this same logic in a user interface for faculty and students.

---

## Step 1 — Install the Anthropic library

In [ ]:
!pip install anthropic -q

## Step 2 — Paste a real grading rubric

In [ ]:
RUBRIC = """
Brand Strategy Case Study - Grading Rubric (100 pts)

1. Market Analysis (25 pts)
   - Correctly identifies target segment with data-backed justification
   - Realistic market sizing

2. Brand Positioning (25 pts)
   - Clear, differentiated positioning statement
   - Addresses competitive landscape

3. Strategic Rationale (25 pts)
   - Logical connection between analysis and recommendations
   - Acknowledges trade-offs and risks

4. Communication (25 pts)
   - Concise and structured argument
   - Professional vocabulary
"""

## Step 3 — Ask Claude to generate oral exam questions from the rubric

This is the core of OralIQ. Claude does not get a list of topics — it reads the rubric criteria and decides what to ask on its own. That autonomous reasoning is what makes it agentic.

In [ ]:
import anthropic
import json

client = anthropic.Anthropic(api_key="sk-ant-YOUR-KEY-HERE")

response = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1000,
    system="You are a Marketing professor creating oral exam questions that test genuine understanding, not memorization.",
    messages=[{
        "role": "user",
        "content": (
            "Read this rubric and generate 4 oral exam questions - one per criterion.\n"
            "Each question should require a student to think and apply knowledge, not just recall.\n"
            "Assign a realistic examiner role to each: Examiner, Skeptical Client, Supervisor, or Peer Reviewer.\n\n"
            "RUBRIC:\n" + RUBRIC + "\n\n"
            "Return ONLY a JSON array: [{\"question\": \"...\", \"criterion\": \"...\", \"role\": \"...\"}]"
        )
    }]
)

questions = json.loads(response.content[0].text)

print(f"Generated {len(questions)} questions:\n")
for i, q in enumerate(questions, 1):
    print(f"Q{i} [{q['role']}]: {q['question']}")
    print(f"   Tests: {q['criterion']}\n")

## Step 4 — Grade a student's answer against the rubric

The second agentic step: Claude evaluates what the student said, referencing the original criterion it generated the question from.

In [ ]:
student_answer = """
We are targeting millennial professionals aged 25 to 35 in urban markets.
Census data puts this at about 18 million people in the US, growing 12 percent
over 5 years. Competitors focus on Gen Z, so positioning as premium for this
segment fills a clear gap. The risk is higher acquisition cost, but lifetime
value justifies it.
"""

grade_response = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=400,
    system="You are a fair business school professor grading an oral exam response. Be specific.",
    messages=[{
        "role": "user",
        "content": (
            "Grade this oral exam response.\n\n"
            "Question: " + questions[0]["question"] + "\n"
            "Rubric criterion: " + questions[0]["criterion"] + "\n"
            "Student answer: " + student_answer + "\n\n"
            "Return ONLY JSON: {\"score\": 0-10, \"feedback\": \"...\", \"strengths\": \"...\", \"improvements\": \"...\"}"
        )
    }]
)

grade = json.loads(grade_response.content[0].text)
print(f"Score:        {grade['score']}/10")
print(f"Feedback:     {grade['feedback']}")
print(f"Strengths:    {grade['strengths']}")
print(f"Improvements: {grade['improvements']}")

---
## Summary: Why This Is Agentic

| Step | What Claude does autonomously |
|------|------------------------------|
| Step 3 | Reads rubric, decides what questions to ask, assigns examiner roles |
| Step 4 | Reads student answer, scores against specific criterion, writes targeted feedback |

No human chooses the questions or decides how to score. Claude reasons through both steps on its own.

The full OralIQ web app wraps these exact API calls in a user interface so faculty and students can use it without writing any code.